# Phase 4 -- Kill vs Shrink

**RHOAIENG-85715**

A low-priority job runs on 8 GPUs. A high-priority job arrives needing 4.

| | Version A (Kill) | Version B (Shrink) |
|---|---|---|
| **What happens** | Kill low-priority, free all 8 GPUs. High-priority takes 4. Restart low-priority later. | Low-priority gives back 4 GPUs and keeps running on 4. High-priority takes the freed 4. |
| **Low-priority during high-priority** | Idle (not training) | Training at half speed |
| **Low-priority after high-priority** | Restarts on 8 GPUs from checkpoint | Scales back to 8 GPUs |

The difference is the headline number for the whole project.

### Simulation note

Elastic resize is not reachable through Kubeflow Trainer today: `numNodes` can
be patched on a running TrainJob, but the controller has no logic to act on it,
so the JobSet underneath never changes (Phase 2).

The shrink is therefore simulated by killing the job and resubmitting it on
fewer GPUs from its checkpoint. That is faithful, because torchrun tears down
every worker on any change to the world size, so a native resize performs the
same teardown and the same restore. The one thing the simulation adds is pod
recreation, which a native resize would avoid. Container startup is recorded
per submission from pod events so its size can be judged, not assumed.

JIT checkpointing is enabled, which is the SDK default once `output_dir` is
set. A resize is a graceful shutdown, so the job saves on SIGTERM instead of
falling back to its last periodic checkpoint. Both arms get it, so neither is
handicapped, and the cost of a resize stays the restart alone.

### What we measure

1. **GPU drain time** -- how long from "high-priority needs GPUs" to "high-priority starts"
2. **Work lost** -- training steps discarded because they came after the
   checkpoint the next run resumes from. JIT checkpointing is on, so a graceful
   delete saves at the step the job is on and this should be near zero.
3. **Low-priority wall time** -- total time from first submission to final completion
4. **Fleet utilization** -- fraction of the node's 8 GPUs doing useful work

### Prerequisites

- IBM cluster workbench in `dhryshch-elastic-scaling` namespace
- PVC `elastic-scaling-shared` and Secret `hf-token`
- 8 GPUs free on the target node

## 1. Setup

Install the Kubeflow SDK, YAML magic, and the local `elastic_scaling_poc`
package so `train_func` can be imported.

In [ ]:
!pip install --force-reinstall --no-cache-dir -U \
    "kubeflow @ git+https://github.com/opendatahub-io/kubeflow-sdk.git@v0.3.0+rhaiv.2"
!pip install yamlmagic hatchling --index-url https://pypi.org/simple
!pip install --no-deps .. --index-url https://pypi.org/simple
%load_ext yamlmagic

## 2. Configuration

All parameters in one place. The experiment settings at the bottom
control how many steps each job runs and when the preemption happens.

- **low_priority_steps**: total steps for the low-priority job (300)
- **high_priority_steps**: total steps for the high-priority job (100)
- **interrupt_at_step**: checkpoint step where we preempt (100)
- **save_steps**: checkpoint frequency (every 100 steps)

In [ ]:
%%yaml parameters

# Infrastructure
namespace: dhryshch-elastic-scaling
pvc_name: elastic-scaling-shared
hf_secret_name: hf-token
target_node: oai-kft-ibm-jcsbk-gpu-2-8gmgw

# Model & Data
model_id: meta-llama/Llama-3.1-8B
dataset_id: tatsu-lab/alpaca
output_dir: /mnt/kubeflow-checkpoints

# Training (shared)
seq_length: 1024
seed: 42
global_batch_size: 128
per_device_batch_size: 4
lora_r: 16
lora_alpha: 32
warmup_steps_excluded: 20
save_steps: 100

# Phase 4 experiment
low_priority_steps: 300
high_priority_steps: 100
interrupt_at_step: 100

In [ ]:
%load_ext autoreload
%autoreload 2

from elastic_scaling_poc.phase4.train import train_func

print("train_func loaded")

In [ ]:
import os
from pathlib import Path

from kubeflow.common.types import KubernetesBackendConfig
from kubeflow.trainer import TrainerClient
from kubernetes import client as k8s

api_server = os.environ["OPENSHIFT_API_URL"]
token = os.getenv("NOTEBOOK_USER_TOKEN", "")
if not token:
    sa_path = Path("/var/run/secrets/kubernetes.io/serviceaccount/token")
    if sa_path.exists():
        token = sa_path.read_text().strip()
if not token:
    raise RuntimeError(
        "Set NOTEBOOK_USER_TOKEN, or run inside a workbench "
        "with a service-account token."
    )

config = k8s.Configuration()
config.host = api_server
config.api_key = {"authorization": f"Bearer {token}"}
config.verify_ssl = False

client = TrainerClient(
    KubernetesBackendConfig(
        namespace=parameters["namespace"],
        client_configuration=config,
    )
)

In [ ]:
core_api = k8s.CoreV1Api(k8s.ApiClient(config))


def get_container_startup_time(job_name):
    """Time between pod creation and training container start."""
    results = []
    pods = core_api.list_namespaced_pod(
        namespace=parameters["namespace"],
        label_selector=f"jobset.sigs.k8s.io/jobset-name={job_name}",
    )
    for pod in pods.items:
        created = pod.metadata.creation_timestamp
        for status in pod.status.container_statuses or []:
            if status.name != "node":
                continue
            state = status.state.terminated or status.state.running
            if state and state.started_at:
                delta = (state.started_at - created).total_seconds()
                results.append(delta)
                print(f"  {pod.metadata.name}: container startup {delta:.1f}s")
    return results

## 3. Helpers

Job submission, monitoring, and measurement functions. Three things to note:

- **Checkpoint directory is keyed on role, not GPU count.** A low-priority
  job restarting at 4 GPUs finds checkpoints saved at 8 GPUs.
- **Checkpoint names are read, never reconstructed.** JIT writes
  `checkpoint-<step SIGTERM landed on>`, which is not a `save_steps` multiple,
  so `checkpoint_steps_on_disk` has to list the directory.
- **Progress comes from pod logs, not the PVC.** The workbench cannot reliably
  see files the training pods write: its NFS client caches the lookup and
  neither retrying nor forcing a readdir clears it. `wait_for_step` therefore
  reads `[phase4] step=N` from the pod log through the API server. Anything
  that still reads the PVC (`read_if_present`, `checkpoint_steps_on_disk`) runs
  only after the pods are gone, and is retried.

In [ ]:
import json
import os
import shutil
import time

from kubeflow.trainer.options import (
    ContainerOverride,
    Name,
    PodSpecOverride,
    PodTemplateOverride,
    PodTemplateOverrides,
)
from kubeflow.trainer.rhai import TransformersTrainer
from kubeflow.trainer.rhai.transformers import PeriodicCheckpointConfig
from kubeflow_trainer_api.models import IoK8sApimachineryPkgApiResourceQuantity

PACKAGES = ["datasets", "peft", "trl", "nvidia-ml-py"]
STEP_PREFIX = "[phase4] step="  # printed by phase4/train.py on rank 0
CURRENT_JOB = {}  # role -> most recently submitted job name
WORKBENCH_MOUNT = Path("/opt/app-root/src/elastic-scaling-shared")
FLEET_GPUS = 8


def run_dir(role):
    return WORKBENCH_MOUNT / f"phase4-{role}"


def checkpoint_path(role):
    return run_dir(role) / "checkpoints"


def pod_overrides():
    return PodTemplateOverrides(
        PodTemplateOverride(
            target_jobs=["node"],
            spec=PodSpecOverride(
                node_selector={"kubernetes.io/hostname": parameters["target_node"]},
                volumes=[
                    {
                        "name": "dshm",
                        "emptyDir": {
                            "medium": "Memory",
                            "sizeLimit": IoK8sApimachineryPkgApiResourceQuantity("16Gi"),
                        },
                    },
                    {
                        "name": "hf-token",
                        "secret": {"secretName": parameters["hf_secret_name"]},
                    },
                ],
                containers=[
                    ContainerOverride(
                        name="node",
                        volume_mounts=[
                            {"name": "dshm", "mountPath": "/dev/shm"},
                            {"name": "hf-token", "mountPath": "/mnt/hf-token", "readOnly": True},
                        ],
                    ),
                ],
            ),
        )
    )


def submit_job(name, role, gpus, num_nodes=1, **overrides):
    ws = num_nodes * gpus
    ckpt_name = f"phase4-{role}"
    trainer = TransformersTrainer(
        func=train_func,
        func_args={**parameters, "role": role, **overrides},
        num_nodes=num_nodes,
        resources_per_node={"nvidia.com/gpu": gpus},
        packages_to_install=PACKAGES,
        output_dir=f"pvc://{parameters['pvc_name']}/{ckpt_name}/checkpoints",
        periodic_checkpoint_config=PeriodicCheckpointConfig(
            save_strategy="steps",
            save_steps=parameters["save_steps"],
        ),
    )
    # On by default once output_dir is set. A resize is a graceful shutdown,
    # so SIGTERM checkpointing keeps the cost of a resize to the restart alone.
    trainer.enable_jit_checkpoint = True
    runtime = client.backend.get_runtime("torch-distributed")
    job_name = client.train(
        trainer=trainer,
        runtime=runtime,
        options=[pod_overrides(), Name(name)],
    )
    CURRENT_JOB[role] = job_name
    print(f"Submitted {job_name} ({ws} GPUs, role={role})")
    return job_name


def watch_job(name, poll_s=15, timeout_s=7200):
    seen, start = None, time.time()
    while time.time() - start < timeout_s:
        status = client.get_job(name).status
        if status != seen:
            print(f"  [{int(time.time() - start):5d}s] {status}")
            seen = status
        if status in ("Complete", "Failed"):
            return status
        time.sleep(poll_s)
    return "Timeout"


def require_complete(status, job_name):
    if status != "Complete":
        raise RuntimeError(f"{job_name} did not complete: {status}")


def current_step(job_name, tail_lines=200):
    """Highest step rank 0 has logged, or None if it has not started training.

    Reads pod logs through the API server, which is the only progress channel
    the workbench can reach. See the note above cell 3. Only the tail is
    fetched: markers arrive in order, so the newest is always near the end.
    """
    pods = core_api.list_namespaced_pod(
        namespace=parameters["namespace"],
        label_selector=f"jobset.sigs.k8s.io/jobset-name={job_name}",
    )
    highest = None
    for pod in pods.items:
        if pod.status.phase not in ("Running", "Succeeded"):
            continue
        try:
            log = core_api.read_namespaced_pod_log(
                name=pod.metadata.name,
                namespace=parameters["namespace"],
                container="node",
                tail_lines=tail_lines,
            )
        except Exception:
            continue
        for line in log.splitlines():
            # tqdm writes its bar with carriage returns, so the marker often
            # lands mid-line. Find it anywhere, then read the digits after it.
            start = line.find(STEP_PREFIX)
            if start == -1:
                continue
            digits = ""
            for char in line[start + len(STEP_PREFIX):]:
                if not char.isdigit():
                    break
                digits += char
            if not digits:
                continue
            step = int(digits)
            highest = step if highest is None else max(highest, step)
    return highest


def wait_for_step(role, gpus, step, job_name=None, poll_s=10, timeout_s=3600):
    """Wait until rank 0 has logged a step past `step`.

    Matches the exact marker phase4/train.py prints, so a tqdm "N/M" bar cannot
    be mistaken for progress. `gpus` is unused and kept for call compatibility.
    """
    job_name = job_name or CURRENT_JOB[role]
    start = time.time()
    while time.time() - start < timeout_s:
        reached = current_step(job_name)
        if reached is not None and reached > step:
            print(f"step {step} passed (rank 0 at step {reached})")
            return True
        time.sleep(poll_s)
    print(f"Timeout waiting for step {step} in {job_name}")
    return False


def wait_for_pods_gone(job_name, poll_s=5, timeout_s=300):
    start = time.time()
    while time.time() - start < timeout_s:
        pods = core_api.list_namespaced_pod(
            namespace=parameters["namespace"],
            label_selector=f"jobset.sigs.k8s.io/jobset-name={job_name}",
        )
        if not pods.items:
            elapsed = int(time.time() - start)
            print(f"Pods released after {elapsed}s")
            return True
        time.sleep(poll_s)
    print(f"Timeout waiting for {job_name} pods to terminate")
    return False


WORLD_SIZES = (4, 8)  # GPU counts this experiment submits jobs at


def expected_run_files(role, world_sizes=WORLD_SIZES):
    """Output paths a run of `role` can produce, built explicitly.

    Paths are constructed rather than globbed so a listing is never needed.
    """
    rd = run_dir(role)
    for ws in world_sizes:
        yield rd / f"metrics-{ws}gpu-rank0.jsonl"
        yield rd / f"startup_timing_{ws}gpu.json"


def read_if_present(path, retries=1, retry_s=5):
    """Read a file, or None if absent. Retries outlast a cached negative lookup."""
    for attempt in range(retries):
        try:
            return path.read_text()
        except FileNotFoundError:
            if attempt + 1 < retries:
                time.sleep(retry_s)
    return None


def checkpoint_steps_on_disk(role, retries=3, retry_s=5):
    """Checkpoint steps present on the PVC, ascending.

    JIT checkpointing writes at whatever step SIGTERM was handled on, so names
    cannot be reconstructed from save_steps and the directory has to be read.
    Read after wait_for_pods_gone, so a JIT checkpoint written on SIGTERM has
    already landed.
    """
    ckpt_dir = checkpoint_path(role)
    prefix = "checkpoint-"
    for attempt in range(retries):
        try:
            steps = sorted(
                int(name[len(prefix):])
                for name in os.listdir(ckpt_dir)
                if name.startswith(prefix) and name[len(prefix):].isdigit()
            )
            if steps:
                return steps
        except FileNotFoundError:
            pass
        if attempt + 1 < retries:
            time.sleep(retry_s)
    return []


def clean_checkpoints(role):
    """Remove every checkpoint for this role, JIT ones included.

    A JIT checkpoint sits at an arbitrary step, so leaving one behind would let
    the next arm resume from the previous arm's progress.
    """
    ckpt_dir = checkpoint_path(role)
    removed = []
    for step in checkpoint_steps_on_disk(role, retries=1):
        try:
            shutil.rmtree(ckpt_dir / f"checkpoint-{step}")
            removed.append(f"checkpoint-{step}")
        except FileNotFoundError:
            pass
    print(f"{ckpt_dir}: removed {len(removed)} checkpoint(s) {removed}")


def clean_metrics(role):
    """Delete this role's metrics and timing files by name."""
    for path in expected_run_files(role):
        try:
            path.unlink()
        except FileNotFoundError:
            pass


def latest_checkpoint_step(role):
    """Step the next run will resume from: the highest checkpoint on the PVC.

    Call after wait_for_pods_gone. JIT saves at the next on_step_end after
    SIGTERM and the process then exits, so once the pods are gone the
    checkpoint is already written.
    """
    steps = checkpoint_steps_on_disk(role)
    return steps[-1] if steps else 0


def measure_work_lost(role, gpus, checkpoint_step):
    """Steps trained past `checkpoint_step`, from rank 0's metrics file.

    `checkpoint_step` is the step actually resumed from, so pass the value from
    latest_checkpoint_step rather than a save_steps boundary.
    """
    text = read_if_present(run_dir(role) / f"metrics-{gpus}gpu-rank0.jsonl")
    if text is None:
        print(f"No metrics file for {role} {gpus}gpu")
        return 0
    last_step = checkpoint_step
    for line in text.strip().split("\n"):
        if not line:
            continue
        try:
            record = json.loads(line)
        except json.JSONDecodeError:
            continue
        if record.get("type") == "step":
            last_step = max(last_step, record["step"])
    lost = last_step - checkpoint_step
    print(f"Work lost for {role}: {lost} steps (last trained: {last_step}, checkpoint: {checkpoint_step})")
    return lost


In [ ]:
# Output layout under results/phase4:
#   phase4_summary.json
#   version_a/events.json
#   version_a/<role>_startup_timing_<n>gpu.json
#   version_a/<role>_metrics-<n>gpu-rank0.jsonl
#   version_b/...
#
# Only rank 0 records metrics, so only rank 0 files are collected.

RESULTS_DIR = Path("../results/phase4")


def version_dir(version):
    """Directory for one arm of the experiment, created if missing."""
    path = RESULTS_DIR / f"version_{version.lower()}"
    path.mkdir(parents=True, exist_ok=True)
    return path


def save_version_results(version, event_log, ran_at, retries=13, retry_s=5):
    """Copy this arm's outputs off the PVC, then write the event log.

    `ran_at` maps role to the GPU counts that arm actually submitted, so only
    files that should exist are waited on. Paths are constructed rather than
    globbed, and each read retries for up to retries * retry_s seconds because
    the workbench may not see a file the moment a pod writes it.
    """
    dest_dir = version_dir(version)
    for role, world_sizes in ran_at.items():
        for src in expected_run_files(role, world_sizes):
            text = read_if_present(src, retries=retries, retry_s=retry_s)
            if text is None:
                print(f"MISSING after {retries * retry_s}s: {src}")
                continue
            dest = dest_dir / f"{role}_{src.name}"
            dest.write_text(text)
            print(f"Saved {dest.relative_to(RESULTS_DIR)}")

    events_path = dest_dir / "events.json"
    events_path.write_text(json.dumps(event_log.to_dict(), indent=2))
    print(f"Saved {events_path.relative_to(RESULTS_DIR)}")

## 4. Event recorder

Timestamps every action with `time.time()` so the analysis can compute
wall times, drain times, and GPU-hours. Container startup times are
stored on events for the elastic-adjustment calculation.

In [ ]:
import time


class EventLog:
    def __init__(self, version):
        self.version = version
        self.events = []

    def record(self, label, **extra):
        t = time.time()
        entry = {"label": label, "time": t, **extra}
        self.events.append(entry)
        ts = time.strftime("%H:%M:%S", time.localtime(t))
        print(f"[{ts}] {label}")
        return t

    def elapsed(self, start_label, end_label):
        start = next(e["time"] for e in self.events if e["label"] == start_label)
        end = next(e["time"] for e in self.events if e["label"] == end_label)
        return end - start

    def to_dict(self):
        return {"version": self.version, "events": self.events}

---

## 5. Version A -- Kill (how it works today)

```
Timeline:
[====== low 8GPU ======]  KILL  [---- idle ----]  [=== high 4GPU ===]  [==== low restart 8GPU ====]
                            ^                                              ^
                        preemption                                    GPUs freed
```

**Steps:**
1. Low-priority trains on 8 GPUs until checkpoint at step 100
2. Record `high_priority_arrival`, then kill low-priority
3. Measure work lost (steps past checkpoint before kill landed)
4. High-priority runs on 4 GPUs -- low-priority is completely idle
5. After high-priority finishes, restart low-priority on 8 GPUs from checkpoint

**Expected output:** timestamps for each event, container startup times,
work lost count, total wall time.

In [ ]:
clean_checkpoints("low-priority")
clean_checkpoints("high-priority")
clean_metrics("low-priority")
clean_metrics("high-priority")

log_a = EventLog("A")

# Step 1: Low-priority on 8 GPUs
low_priority_job_a = submit_job(
    "phase4a-low",
    role="low-priority",
    gpus=8,
    max_steps=parameters["low_priority_steps"],
)
log_a.record("low_priority_submitted", gpus=8)

if not wait_for_step("low-priority", 8, parameters["interrupt_at_step"]):
    raise RuntimeError("Checkpoint never appeared, aborting")
log_a.record("low_priority_checkpoint_reached", step=parameters["interrupt_at_step"])

# Step 2: Preemption trigger - high-priority needs GPUs
log_a.record("high_priority_arrival")

print("\nContainer startup (low-priority initial):")
low_priority_container_startup_a = get_container_startup_time(low_priority_job_a)
log_a.record("low_priority_initial_container_startup",
             container_startup_s=max(low_priority_container_startup_a) if low_priority_container_startup_a else 0)

client.delete_job(name=low_priority_job_a)
log_a.record("low_priority_killed")
wait_for_pods_gone(low_priority_job_a)
log_a.record("low_priority_gpus_released")

resume_step_a = latest_checkpoint_step("low-priority")
work_lost_a = measure_work_lost("low-priority", 8, resume_step_a)
log_a.record("work_lost_measured", steps_lost=work_lost_a,
             resumed_from_step=resume_step_a)

# Step 3: High-priority on 4 GPUs (low-priority is idle)
high_priority_job_a = submit_job(
    "phase4a-high",
    role="high-priority",
    gpus=4,
    max_steps=parameters["high_priority_steps"],
)
log_a.record("high_priority_submitted", gpus=4)

high_priority_status_a = watch_job(high_priority_job_a)
require_complete(high_priority_status_a, "phase4a-high")
log_a.record("high_priority_complete")

print("\nContainer startup (high-priority):")
high_priority_container_startup_a = get_container_startup_time(high_priority_job_a)
log_a.record("high_priority_container_startup",
             container_startup_s=max(high_priority_container_startup_a) if high_priority_container_startup_a else 0)

# Step 4: Low-priority restarts on 8 GPUs (all free now)
low_priority_restart_a = submit_job(
    "phase4a-low-restart",
    role="low-priority",
    gpus=8,
    max_steps=parameters["low_priority_steps"],
)
log_a.record("low_priority_restarted", gpus=8)

low_priority_status_a = watch_job(low_priority_restart_a)
require_complete(low_priority_status_a, "phase4a-low-restart")
log_a.record("low_priority_complete")

print("\nContainer startup (low-priority restart):")
low_priority_restart_container_a = get_container_startup_time(low_priority_restart_a)
log_a.record("low_priority_restart_container_startup",
             container_startup_s=max(low_priority_restart_container_a) if low_priority_restart_container_a else 0)

print("\nVersion A complete.")
print(f"  Total wall time: {log_a.elapsed('low_priority_submitted', 'low_priority_complete'):.0f}s")
print(f"  GPU drain time: {log_a.elapsed('high_priority_arrival', 'high_priority_submitted'):.0f}s")
print(f"  Work lost: {work_lost_a} steps")

### Save Version A, clean up for Version B

Copy timing files and metrics from the PVC, save the event log, delete
jobs, and clear checkpoints so Version B starts clean.

In [ ]:
save_version_results("a", log_a,
                     ran_at={"low-priority": [8], "high-priority": [4]})

# Clean up jobs
for name in ["phase4a-low", "phase4a-high", "phase4a-low-restart"]:
    try:
        client.delete_job(name=name)
        print(f"Deleted {name}")
    except Exception as e:
        print(f"Skip {name}: {e}")

# Clean checkpoints for Version B
clean_checkpoints("low-priority")
clean_checkpoints("high-priority")
clean_metrics("low-priority")
clean_metrics("high-priority")

---

## 6. Version B -- Shrink (the proposal)

```
Timeline:
[====== low 8GPU ======]  SHRINK  [=== low 4GPU === | === high 4GPU ===]  SCALE BACK  [== low 8GPU ==]
                            ^           concurrent on same node              ^
                        preemption                                       GPUs freed
```

**Steps:**
1. Low-priority trains on 8 GPUs until checkpoint at step 100
2. Record `high_priority_arrival`, then kill low-priority (simulates shrink)
3. Submit low-priority on 4 GPUs AND high-priority on 4 GPUs concurrently
4. Wait for high-priority to finish (low-priority keeps training the whole time)
5. Scale low-priority back to 8 GPUs from its latest checkpoint

**On step 5.** Do not wait for the next periodic checkpoint. With
`save_steps=100` and `max_steps=300` the next one is the end of the run, so
waiting means the scale-back never happens and only half the proposal gets
tested. JIT checkpointing removes the need to wait: deleting the job sends
SIGTERM, the job saves at the step it is on, and the scaled-back job resumes
from there. `scaleback_work_lost_measured` records whatever that costs.

**Edge case handled:** if low-priority finishes during the concurrent phase
there is nothing to scale back.

In [ ]:
log_b = EventLog("B")

# Step 1: Low-priority on 8 GPUs
low_priority_job_b = submit_job(
    "phase4b-low",
    role="low-priority",
    gpus=8,
    max_steps=parameters["low_priority_steps"],
)
log_b.record("low_priority_submitted", gpus=8)

if not wait_for_step("low-priority", 8, parameters["interrupt_at_step"]):
    raise RuntimeError("Checkpoint never appeared, aborting")
log_b.record("low_priority_checkpoint_reached", step=parameters["interrupt_at_step"])

# Step 2: Preemption trigger
log_b.record("high_priority_arrival")

print("\nContainer startup (low-priority initial):")
low_priority_container_startup_b = get_container_startup_time(low_priority_job_b)
log_b.record("low_priority_initial_container_startup",
             container_startup_s=max(low_priority_container_startup_b) if low_priority_container_startup_b else 0)

client.delete_job(name=low_priority_job_b)
log_b.record("low_priority_shrink_started")
wait_for_pods_gone(low_priority_job_b)
log_b.record("low_priority_gpus_released")

initial_resume_step_b = latest_checkpoint_step("low-priority")
initial_work_lost_b = measure_work_lost("low-priority", 8, initial_resume_step_b)
log_b.record("initial_work_lost_measured", steps_lost=initial_work_lost_b,
             resumed_from_step=initial_resume_step_b)

# Step 3: Both jobs run concurrently on 4 GPUs each
log_b.record("concurrent_phase_started")

low_priority_shrunk_b = submit_job(
    "phase4b-low-shrunk",
    role="low-priority",
    gpus=4,
    max_steps=parameters["low_priority_steps"],
)
log_b.record("low_priority_shrunk_submitted", gpus=4)

high_priority_job_b = submit_job(
    "phase4b-high",
    role="high-priority",
    gpus=4,
    max_steps=parameters["high_priority_steps"],
)
log_b.record("high_priority_submitted", gpus=4)

# Wait for high-priority to finish (low-priority keeps training)
high_priority_status_b = watch_job(high_priority_job_b)
require_complete(high_priority_status_b, "phase4b-high")
log_b.record("high_priority_complete")

print("\nContainer startup (high-priority):")
high_priority_container_startup_b = get_container_startup_time(high_priority_job_b)
log_b.record("high_priority_container_startup",
             container_startup_s=max(high_priority_container_startup_b) if high_priority_container_startup_b else 0)

# Step 4: Scale back to 8 GPUs.
#
# Do not wait for the next periodic checkpoint. With save_steps=100 and
# max_steps=300 the next one is the end of the run, so waiting means the
# scale-back never happens. JIT checkpointing makes waiting unnecessary: the
# delete below sends SIGTERM, the job saves at the step it is on, and the
# scaled-back job resumes from there.
low_priority_status_check = client.get_job("phase4b-low-shrunk").status
scaleback_work_lost_b = 0

if low_priority_status_check in ("Complete", "Failed"):
    require_complete(low_priority_status_check, "phase4b-low-shrunk")
    log_b.record("low_priority_complete", note="finished during concurrent phase")
    print("Low-priority finished during concurrent phase, nothing to scale back")
else:
    print("\nContainer startup (low-priority shrunk):")
    low_priority_shrunk_container_b = get_container_startup_time(low_priority_shrunk_b)
    log_b.record("low_priority_shrunk_container_startup",
                 container_startup_s=max(low_priority_shrunk_container_b) if low_priority_shrunk_container_b else 0)

    client.delete_job(name=low_priority_shrunk_b)
    log_b.record("low_priority_scale_back_started")
    wait_for_pods_gone(low_priority_shrunk_b)
    log_b.record("low_priority_scale_back_gpus_released")

    resume_from = latest_checkpoint_step("low-priority")
    print(f"Scaling back to 8 GPUs from checkpoint-{resume_from}")
    scaleback_work_lost_b = measure_work_lost("low-priority", 4, resume_from)
    log_b.record("scaleback_work_lost_measured", steps_lost=scaleback_work_lost_b,
                 resumed_from_step=resume_from)

    low_priority_scaled_b = submit_job(
        "phase4b-low-scaled",
        role="low-priority",
        gpus=8,
        max_steps=parameters["low_priority_steps"],
    )
    log_b.record("low_priority_scaled_back_submitted", gpus=8)

    low_priority_status_b = watch_job(low_priority_scaled_b)
    require_complete(low_priority_status_b, "phase4b-low-scaled")
    log_b.record("low_priority_complete")

    print("\nContainer startup (low-priority scaled back):")
    low_priority_scaled_container_b = get_container_startup_time(low_priority_scaled_b)
    log_b.record("low_priority_scaled_container_startup",
                 container_startup_s=max(low_priority_scaled_container_b) if low_priority_scaled_container_b else 0)

total_work_lost_b = initial_work_lost_b + scaleback_work_lost_b

print("\nVersion B complete.")
print(f"  Total wall time: {log_b.elapsed('low_priority_submitted', 'low_priority_complete'):.0f}s")
print(f"  GPU drain time: {log_b.elapsed('high_priority_arrival', 'high_priority_submitted'):.0f}s")
print(f"  Work lost: {total_work_lost_b} steps (initial: {initial_work_lost_b}, scale-back: {scaleback_work_lost_b})")

### Save Version B results

Same pattern as Version A: copy files from PVC, save event log, delete jobs.

In [ ]:
save_version_results("b", log_b,
                     ran_at={"low-priority": [8, 4], "high-priority": [4]})

for name in ["phase4b-low", "phase4b-high", "phase4b-low-shrunk", "phase4b-low-scaled"]:
    try:
        client.delete_job(name=name)
        print(f"Deleted {name}")
    except Exception as e:
        print(f"Skip {name}: {e}")

---

## 7. Analysis

Everything below reads from saved files in `../results/phase4/`, so it
can be re-run without repeating the experiments.

### Headline comparison

Six metrics side by side. The first four come from the Jira ticket. The
last two (gap time and restart overhead) break down where the non-training
time goes.

- **GPU drain time**: arrival to high-priority submission (kill + pod teardown + resubmit)
- **High-priority run time**: submission to completion (includes container startup + training)
- **Work lost**: steps trained past the last checkpoint, discarded on kill
- **Low-priority wall time**: first submission to final completion
- **Gap time**: time between kill and resubmission (excludes restart overhead)
- **Restart overhead**: container startup + model load + checkpoint restore (from timing files)

In [ ]:
import json

import pandas as pd

# Read the saved event logs so this section runs without the experiment cells.
log_a_data = json.loads((version_dir("a") / "events.json").read_text())
log_b_data = json.loads((version_dir("b") / "events.json").read_text())


def get_event(events, label):
    return next((e for e in events["events"] if e["label"] == label), None)


def elapsed(events, start, end):
    s = get_event(events, start)
    e = get_event(events, end)
    if s and e:
        return e["time"] - s["time"]
    return None


def load_timing(version, role, gpus):
    path = version_dir(version) / f"{role}_startup_timing_{gpus}gpu.json"
    if path.exists():
        return json.loads(path.read_text())
    return None


def first_step_elapsed(version, role, gpus):
    """Script-start to first training step, from the startup timing file."""
    timing = load_timing(version, role, gpus)
    if not timing:
        return None
    for entry in timing.get("timeline", []):
        if entry["label"] == "first_step_complete":
            return entry["elapsed_s"]
    return None


def container_startup_from_event(events, label):
    event = get_event(events, label)
    return event["container_startup_s"] if event else 0


# 1. GPU drain time (arrival to high-priority submission)
drain_time_a = elapsed(log_a_data, "high_priority_arrival", "high_priority_submitted")
drain_time_b = elapsed(log_b_data, "high_priority_arrival", "high_priority_submitted")

# High-priority run time (submission to completion, includes startup + training)
high_priority_run_a = elapsed(log_a_data, "high_priority_submitted", "high_priority_complete")
high_priority_run_b = elapsed(log_b_data, "high_priority_submitted", "high_priority_complete")

# 2. Low-priority work lost (measured from metrics files)
work_lost_event_a = get_event(log_a_data, "work_lost_measured")
measured_work_lost_a = work_lost_event_a["steps_lost"] if work_lost_event_a else 0

initial_lost_event_b = get_event(log_b_data, "initial_work_lost_measured")
scaleback_lost_event_b = get_event(log_b_data, "scaleback_work_lost_measured")
measured_work_lost_b = (
    (initial_lost_event_b["steps_lost"] if initial_lost_event_b else 0)
    + (scaleback_lost_event_b["steps_lost"] if scaleback_lost_event_b else 0)
)

# 3. Low-priority total completion time
low_priority_total_a = elapsed(log_a_data, "low_priority_submitted", "low_priority_complete")
low_priority_total_b = elapsed(log_b_data, "low_priority_submitted", "low_priority_complete")

# 4. Low-priority gap time (kill to resubmission, excludes restart overhead)
gap_time_a = elapsed(log_a_data, "low_priority_killed", "low_priority_restarted")

shrink_gap_b = elapsed(log_b_data, "low_priority_shrink_started", "low_priority_shrunk_submitted")
scaleback_gap_b = elapsed(log_b_data, "low_priority_scale_back_started", "low_priority_scaled_back_submitted") if get_event(log_b_data, "low_priority_scale_back_started") else 0
gap_time_b = (shrink_gap_b or 0) + (scaleback_gap_b or 0)

# Restart overhead: container startup + script-to-first-step
restart_container_a = container_startup_from_event(log_a_data, "low_priority_restart_container_startup")
restart_script_a = first_step_elapsed("A", "low-priority", 8) or 0
restart_overhead_a = restart_container_a + restart_script_a

shrunk_container_b = container_startup_from_event(log_b_data, "low_priority_shrunk_container_startup")
shrunk_script_b = first_step_elapsed("B", "low-priority", 4) or 0
scaled_container_b = container_startup_from_event(log_b_data, "low_priority_scaled_container_startup")
scaled_script_b = first_step_elapsed("B", "low-priority", 8) or 0
restart_overhead_b = (shrunk_container_b + shrunk_script_b) + (scaled_container_b + scaled_script_b)

# Container startup adjustment for true elastic scaling
container_adjustment_b = shrunk_container_b + scaled_container_b

print("=" * 60)
print("PHASE 4 RESULTS: KILL vs SHRINK")
print("=" * 60)
print()

comparison = pd.DataFrame({
    "Metric": [
        "GPU drain time (s)",
        "High-priority run time (s)",
        "Low-priority work lost (steps)",
        "Low-priority total wall time (s)",
        "Low-priority gap time (s)",
        "Low-priority restart overhead (s)",
    ],
    "Version A (Kill)": [
        f"{drain_time_a:.0f}" if drain_time_a else "n/a",
        f"{high_priority_run_a:.0f}" if high_priority_run_a else "n/a",
        measured_work_lost_a,
        f"{low_priority_total_a:.0f}" if low_priority_total_a else "n/a",
        f"{gap_time_a:.0f}" if gap_time_a else "n/a",
        f"{restart_overhead_a:.0f}" if restart_overhead_a else "n/a",
    ],
    "Version B (Shrink)": [
        f"{drain_time_b:.0f}" if drain_time_b else "n/a",
        f"{high_priority_run_b:.0f}" if high_priority_run_b else "n/a",
        measured_work_lost_b,
        f"{low_priority_total_b:.0f}" if low_priority_total_b else "n/a",
        f"{gap_time_b:.0f}" if gap_time_b else "n/a",
        f"{restart_overhead_b:.0f}" if restart_overhead_b else "n/a",
    ],
})
print(comparison.to_string(index=False))

if low_priority_total_a and low_priority_total_b:
    saving_s = low_priority_total_a - low_priority_total_b
    saving_pct = (saving_s / low_priority_total_a) * 100
    print(f"\nShrinking saves {saving_s:.0f}s ({saving_pct:.1f}%) on low-priority wall time.")

if container_adjustment_b > 0:
    adjusted_total_b = low_priority_total_b - container_adjustment_b
    adjusted_saving_s = low_priority_total_a - adjusted_total_b
    adjusted_saving_pct = (adjusted_saving_s / low_priority_total_a) * 100
    print(f"\nTrue elastic adjustment (no container startup for shrink/scale-back): -{container_adjustment_b:.0f}s")
    print(f"Adjusted Version B wall time: {adjusted_total_b:.0f}s (saves {adjusted_saving_s:.0f}s / {adjusted_saving_pct:.1f}%)")

### GPU-hours and fleet utilization

Allocated GPU-hours counts only GPUs actively assigned to a job. Fleet
utilization divides that by the total available GPU-hours (8 GPUs for
the full wall time). The difference is idle GPU-hours -- GPUs that
existed on the node but did no work. This is where shrink's advantage
shows: Version A leaves 4 GPUs idle during high-priority's run, Version
B keeps all 8 busy.

In [ ]:
# GPU-hours = sum of (GPU count * duration in hours) for each segment

def gpu_hours_a():
    segments = []
    duration = elapsed(log_a_data, "low_priority_submitted", "low_priority_killed")
    if duration:
        segments.append(("Low-priority initial (8 GPU)", 8, duration))

    duration = elapsed(log_a_data, "high_priority_submitted", "high_priority_complete")
    if duration:
        segments.append(("High-priority (4 GPU)", 4, duration))

    duration = elapsed(log_a_data, "low_priority_restarted", "low_priority_complete")
    if duration:
        segments.append(("Low-priority restart (8 GPU)", 8, duration))

    return segments


def gpu_hours_b():
    segments = []
    duration = elapsed(log_b_data, "low_priority_submitted", "low_priority_shrink_started")
    if duration:
        segments.append(("Low-priority initial (8 GPU)", 8, duration))

    # The shrunk job runs until it is scaled back, or until it completes if the
    # scale-back never happened. Ending this segment at high_priority_complete
    # undercounts whenever the job outlives the high-priority one.
    if get_event(log_b_data, "low_priority_scale_back_started"):
        shrunk_end = "low_priority_scale_back_started"
    else:
        shrunk_end = "low_priority_complete"
    duration = elapsed(log_b_data, "low_priority_shrunk_submitted", shrunk_end)
    if duration:
        segments.append(("Low-priority shrunk (4 GPU)", 4, duration))

    duration = elapsed(log_b_data, "high_priority_submitted", "high_priority_complete")
    if duration:
        segments.append(("High-priority (4 GPU)", 4, duration))

    if get_event(log_b_data, "low_priority_scaled_back_submitted"):
        duration = elapsed(log_b_data, "low_priority_scaled_back_submitted", "low_priority_complete")
        if duration:
            segments.append(("Low-priority scaled back (8 GPU)", 8, duration))

    return segments


print("Version A GPU-hours:")
total_gpu_hours_a = 0
for label, gpus, duration in gpu_hours_a():
    gpu_h = gpus * duration / 3600
    total_gpu_hours_a += gpu_h
    print(f"  {label}: {duration:.0f}s = {gpu_h:.2f} GPU-h")
print(f"  Total allocated: {total_gpu_hours_a:.2f} GPU-h")

print(f"\nVersion B GPU-hours:")
total_gpu_hours_b = 0
for label, gpus, duration in gpu_hours_b():
    gpu_h = gpus * duration / 3600
    total_gpu_hours_b += gpu_h
    print(f"  {label}: {duration:.0f}s = {gpu_h:.2f} GPU-h")
print(f"  Total allocated: {total_gpu_hours_b:.2f} GPU-h")

if total_gpu_hours_a and total_gpu_hours_b:
    diff = total_gpu_hours_a - total_gpu_hours_b
    print(f"\n  Allocated difference: {diff:+.2f} GPU-h ({diff / total_gpu_hours_a * 100:+.1f}%)")

# Fleet utilization: the node has FLEET_GPUS for the entire experiment wall time
print(f"\nFleet utilization ({FLEET_GPUS} GPUs on node):")
if low_priority_total_a:
    fleet_hours_a = FLEET_GPUS * low_priority_total_a / 3600
    idle_hours_a = fleet_hours_a - total_gpu_hours_a
    utilization_a = total_gpu_hours_a / fleet_hours_a * 100
    print(f"  Version A: {utilization_a:.1f}% ({idle_hours_a:.2f} GPU-h idle)")

if low_priority_total_b:
    fleet_hours_b = FLEET_GPUS * low_priority_total_b / 3600
    idle_hours_b = fleet_hours_b - total_gpu_hours_b
    utilization_b = total_gpu_hours_b / fleet_hours_b * 100
    print(f"  Version B: {utilization_b:.1f}% ({idle_hours_b:.2f} GPU-h idle)")

if low_priority_total_a and low_priority_total_b:
    print(f"  Idle GPU-hours saved by shrink: {idle_hours_a - idle_hours_b:.2f}")

### Step time contention

During Version B's concurrent phase, two 4-GPU jobs share one 8-GPU
node. This section checks whether sharing slows them down compared to
the Phase 1 baseline where 4 GPUs had the node to themselves. A
slowdown here would reduce shrink's advantage.

In [ ]:
# Load step-level metrics from Version B concurrent phase
def load_steps(version, role, gpus):
    path = version_dir(version) / f"{role}_metrics-{gpus}gpu-rank0.jsonl"
    if not path.exists():
        print(f"Not found: {path}")
        return pd.DataFrame()
    records = []
    for line in path.read_text().strip().split("\n"):
        if line:
            r = json.loads(line)
            if r.get("type") == "step" and not r.get("warmup"):
                records.append(r)
    return pd.DataFrame(records)


# Phase 1 baselines for comparison
phase1_file = Path("../results/phase1/summaries.json")
if phase1_file.exists():
    phase1 = json.loads(phase1_file.read_text())
    phase1_step_times = {s["world_size"]: s["mean_step_time_s"] for s in phase1}
    print("Phase 1 baseline step times:")
    for ws, st in sorted(phase1_step_times.items()):
        print(f"  {ws} GPU: {st:.4f}s")
else:
    phase1_step_times = {}
    print("Phase 1 baselines not found")

# Version B concurrent phase step times
df_low_priority = load_steps("B", "low-priority", 4)
df_high_priority = load_steps("B", "high-priority", 4)

if not df_low_priority.empty:
    print(f"\nVersion B low-priority (4 GPU, concurrent): mean step = {df_low_priority['step_time_s'].mean():.4f}s")
    if 4 in phase1_step_times:
        slowdown = (df_low_priority["step_time_s"].mean() / phase1_step_times[4] - 1) * 100
        print(f"  vs Phase 1 baseline (4 GPU solo): {slowdown:+.1f}% slowdown from contention")

if not df_high_priority.empty:
    print(f"\nVersion B high-priority (4 GPU, concurrent): mean step = {df_high_priority['step_time_s'].mean():.4f}s")
    if 4 in phase1_step_times:
        slowdown = (df_high_priority["step_time_s"].mean() / phase1_step_times[4] - 1) * 100
        print(f"  vs Phase 1 baseline (4 GPU solo): {slowdown:+.1f}% slowdown from contention")

### Startup timelines

Where the restart time goes (model load, checkpoint restore, etc.) for
each restarted run. In true elastic scaling, container startup would be
zero since pods stay running -- only torchrun restarts within them.

In [ ]:
# Show restart timelines (load_timing defined in the analysis cell above)
for version, label in [("A", "Kill"), ("B", "Shrink")]:
    for role in ["low-priority", "high-priority"]:
        for gpus in [4, 8]:
            t = load_timing(version, role, gpus)
            if t and t.get("resumed_from"):
                print(f"\nVersion {version} ({label}) - {role} {gpus}GPU restart:")
                print(f"  Resumed from: {t['resumed_from']}")
                for entry in t["timeline"]:
                    print(f"  {entry['elapsed_s']:7.3f}s  {entry['label']}")

---

## 8. Save combined summary

Writes `phase4_summary.json` with all metrics, event logs, and the
headline numbers for the research report.

In [ ]:
summary = {
    "experiment": "phase4_kill_vs_shrink",
    "parameters": {
        "low_priority_steps": parameters["low_priority_steps"],
        "high_priority_steps": parameters["high_priority_steps"],
        "interrupt_at_step": parameters["interrupt_at_step"],
        "save_steps": parameters["save_steps"],
        "fleet_gpus": FLEET_GPUS,
    },
    "version_a": {
        "total_wall_time_s": low_priority_total_a,
        "drain_time_s": drain_time_a,
        "gap_time_s": gap_time_a,
        "restart_overhead_s": restart_overhead_a,
        "work_lost_steps": measured_work_lost_a,
        "allocated_gpu_hours": total_gpu_hours_a,
        "fleet_utilization_pct": utilization_a if low_priority_total_a else None,
        "idle_gpu_hours": idle_hours_a if low_priority_total_a else None,
    },
    "version_b": {
        "total_wall_time_s": low_priority_total_b,
        "drain_time_s": drain_time_b,
        "gap_time_s": gap_time_b,
        "restart_overhead_s": restart_overhead_b,
        "work_lost_steps": measured_work_lost_b,
        "allocated_gpu_hours": total_gpu_hours_b,
        "fleet_utilization_pct": utilization_b if low_priority_total_b else None,
        "idle_gpu_hours": idle_hours_b if low_priority_total_b else None,
        "container_startup_adjustment_s": container_adjustment_b,
    },
}

if low_priority_total_a and low_priority_total_b:
    summary["headline"] = {
        "wall_time_saved_s": low_priority_total_a - low_priority_total_b,
        "wall_time_saved_pct": (low_priority_total_a - low_priority_total_b) / low_priority_total_a * 100,
        "idle_gpu_hours_saved": idle_hours_a - idle_hours_b,
        "fleet_utilization_gain_pct": utilization_b - utilization_a,
        "adjusted_wall_time_saved_s": low_priority_total_a - (low_priority_total_b - container_adjustment_b),
    }

summary_path = RESULTS_DIR / "phase4_summary.json"
summary_path.write_text(json.dumps(summary, indent=2))
print(f"Saved to {summary_path.resolve()}")

print("\nAll Phase 4 results:")
for f in sorted(RESULTS_DIR.rglob("*")):
    if f.is_file():
        print(f"  {f.relative_to(RESULTS_DIR)}")

## 9. Cleanup

Delete all jobs from the cluster. Safe to skip if jobs were already
cleaned up in the save cells.

In [ ]:
all_jobs = [
    "phase4a-low", "phase4a-high", "phase4a-low-restart",
    "phase4b-low", "phase4b-high", "phase4b-low-shrunk", "phase4b-low-scaled",
]
for name in all_jobs:
    try:
        client.delete_job(name=name)
        print(f"Deleted {name}")
    except Exception as e:
        print(f"Skip {name}: {e}")